# 08g — Explain (Attention + GNNExplainer, Capacity Revision / GATv2 + pool_anchor)

Post-hoc explainability for `07g`'s trained checkpoints. **Requires `07g`
to have already been run** -- this notebook loads its saved
`{tag}_best_model.pt` / `{tag}_best_model_stats.pt` per scenario, it
does not train anything itself.

**Two techniques, both applied to every explained point:**
1. **Native GATv2 attention-weight extraction** (`explain.extract_attention_weights`)
   -- free (no extra optimization), reads the real per-neighbor attention
   weights the trained model used at each `GATv2Conv` layer.
2. **GNNExplainer-style learned mask** (`explain.run_gnnexplainer`) --
   a small per-point optimization that learns a soft node-importance mask
   (and, for TVG/Unified, an edge-importance mask) that reproduces the
   model's own original prediction as closely as possible while staying
   sparse. Answers "which nodes/edges actually drove THIS prediction,"
   not just "which neighbors got attention at each layer."

Both are implemented directly against this codebase's own
`encoder.forward(data, batch_dict)` signature (`src/explain.py`) rather
than forced through PyTorch Geometric's generic `Explainer` wrapper --
see that module's docstring for why.

**Which points get explained.** For each scenario A-F (G has no GNN
encoder, skipped -- same as `07g` itself), this notebook finds the
SPECIFIC repeat whose weights were kept as `{tag}_best_model.pt`
(`{tag}_best_model_meta.json`'s `repeat` field -- run_scenario_random_repeats
only keeps ONE best-by-val-score repeat's weights per tag, not all 5), then
samples a small number of points **from that repeat's own test split**
(`{tag}_history/repeat{N}_test_predictions.json`) across TP/TN/FP/FN --
so every explained prediction is guaranteed to come from the exact
model whose weights are loaded, not a mismatched repeat/split.

**Normalization.** `07g`'s `train_one_fold` fits per-repeat z-score
stats on that repeat's TRAIN partition only (`ds.fit_normalization`) and
never used to persist them -- `train.py` was extended this session to
save `{tag}_best_model_stats.pt` alongside every `best_model.pt`
specifically so a later notebook like this one can correctly reproduce
the exact input distribution the loaded weights were trained against.
Skipping this step (e.g. explaining raw, unnormalized graphs) would
silently produce meaningless explanations.

**What `pool_anchor` predicts this SHOULD look like.** `07g`'s readout
always concatenates the anchor node's (`ego`/`incident`) own embedding
into the graph vector, by construction (see
`docs/07g_07i_architecture.md` §4). One useful sanity check below: does
GNNExplainer's learned node-importance mask actually rate the anchor
node highly, consistent with the architecture giving it a "free pass"
into every prediction? (Compare against `08i`'s equivalent check, where
`DGCNNReadout` has NO such guarantee.)

GPU recommended for the GNNExplainer optimization loop (small, but runs
once per explained point per branch).

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# TEMP: install locally patched src/ files until pushed to GitHub.
# Skip this cell once the repo itself is updated -- needs explain.py
# (new this session: extract_attention_weights / run_gnnexplainer /
# explain_scenario_point / aggregate_explanations) and train.py
# (best_model_stats.pt persistence), plus models.py, graph_datasets.py,
# unified_graph.py, baseline_features.py, evaluate.py, plot_history.py.
from google.colab import files
import shutil

print("Upload explain.py, train.py, models.py, graph_datasets.py, unified_graph.py, "
      "baseline_features.py, evaluate.py, plot_history.py:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"{REPO_DIR}/src/{fname}")
print("Patched files installed:", list(uploaded.keys()))

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn scipy pyyaml pandas tqdm

In [ ]:
import yaml
from pathlib import Path
import torch

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/eval_capacity_revision.yaml") as f:
    eval_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/model_capacity_revision.yaml") as f:
    model_cfg = yaml.safe_load(f)

CITIES = paths_cfg["cities"]
INTERIM_DIR = Path(paths_cfg["interim_dir"])
COMBINED_PROCESSED_DIR = Path(paths_cfg["processed_dir"])
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])
# MUST match 07g's own dir names exactly -- this notebook only READS
# from CHECKPOINT_DIR/METRICS_DIR, never writes training checkpoints there.
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints_capacity_revision"
METRICS_DIR = OUTPUTS_DIR / "metrics_capacity_revision"
assert CHECKPOINT_DIR.exists(), f"{CHECKPOINT_DIR} not found -- run 07g first."
EXPLAIN_DIR = OUTPUTS_DIR / "explain_capacity_revision"
EXPLAIN_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
HEAD_DEPTH = model_cfg.get("head_depth", "mlp2")

# How many points per confusion-matrix category (TP/TN/FP/FN) to explain,
# per scenario -- keep small, GNNExplainer runs its own small optimization
# loop per point per branch. 2 per category x 4 categories x 6 scenarios
# (C/D/E count as 2 branches each) = a manageable number of optimization
# runs; raise once you've confirmed the pipeline works end to end.
N_PER_CATEGORY = 2
GNNEXPLAINER_EPOCHS = 100
GNNEXPLAINER_LR = 0.05
EXPLAIN_SEED = 42

print("Device:", device, "| head_depth:", HEAD_DEPTH)
print("Reading checkpoints from:", CHECKPOINT_DIR)
print("Writing explanations to:", EXPLAIN_DIR)
print(f"N_PER_CATEGORY={N_PER_CATEGORY}, gnnexplainer_epochs={GNNEXPLAINER_EPOCHS}")

In [ ]:
import json
import copy
import random
import pandas as pd
import graph_datasets as ds
import models
import explain
import unified_graph as ug

SVG_DIR = COMBINED_PROCESSED_DIR / "svg_graphs"
TVG_DIR = COMBINED_PROCESSED_DIR / "tvg_graphs"
INDEX_PATH = COMBINED_PROCESSED_DIR / "dataset_index.parquet"
index_df = pd.read_parquet(INDEX_PATH)
assert "city" in index_df.columns, (
    f"'{INDEX_PATH}' has no 'city' column -- this notebook needs 05's combined, "
    "multi-city dataset_index.parquet, not a single-city index.")
print(f"Dataset: {len(index_df)} points available for lookup")

_ref_cache_dir = INTERIM_DIR / "osm_cache" / CITIES[0]
with open(_ref_cache_dir / "highway_vocab.json") as f:
    HIGHWAY_VOCAB_SIZE = len(json.load(f))
with open(_ref_cache_dir / "building_type_vocab.json") as f:
    BUILDING_TYPE_VOCAB_SIZE = len(json.load(f))
print(f"Unified vocab (post-04b): highway={HIGHWAY_VOCAB_SIZE}, building_type={BUILDING_TYPE_VOCAB_SIZE}")

# Must be IDENTICAL to 07g's own svg_kwargs/tvg_kwargs -- these define the
# model architecture that {tag}_best_model.pt's state_dict was saved
# from; any mismatch (wrong hidden_dim, wrong embed dims, ...) will fail
# to load or silently misalign weights.
svg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("svg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   signage_vocab=5, light_pole_vocab=4, road_marking_vocab=2,
                   cat_embed_dim=model_cfg.get("cat_embed_dim", 4))
tvg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("tvg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   building_type_vocab=BUILDING_TYPE_VOCAB_SIZE, highway_vocab=HIGHWAY_VOCAB_SIZE,
                   building_type_embed_dim=model_cfg.get("building_type_embed_dim", 32),
                   highway_embed_dim=model_cfg.get("highway_embed_dim", 8))
FUSION_DIM = model_cfg.get("fusion_dim", 256)
HEAD_HIDDEN = model_cfg.get("head_hidden", 256)
HEAD_DROPOUT = model_cfg.get("head_dropout", 0.3)
print("svg_kwargs:", svg_kwargs)
print("tvg_kwargs:", tvg_kwargs)

## Helpers: load a trained scenario's model+stats, pick which points to explain

In [ ]:
def load_scenario_model(scenario, use_ablation=False):
    """Reconstructs the exact architecture build_model() would have
    produced during 07g, loads {tag}_best_model.pt's weights and
    {tag}_best_model_stats.pt's normalization stats. Returns
    (model, stats, best_repeat, tag)."""
    tag = f"{scenario}_{HEAD_DEPTH}" + ("_ablation" if use_ablation else "")
    model_path = CHECKPOINT_DIR / f"{tag}_best_model.pt"
    stats_path = CHECKPOINT_DIR / f"{tag}_best_model_stats.pt"
    meta_path = CHECKPOINT_DIR / f"{tag}_best_model_meta.json"
    assert model_path.exists(), f"{model_path} not found -- run 07g's scenario {scenario} cell first."
    assert stats_path.exists(), (
        f"{stats_path} not found -- this checkpoint predates the best_model_stats.pt "
        f"persistence change; rerun 07g's scenario {scenario} cell with the current train.py.")

    model = models.build_model(scenario, fusion_dim=FUSION_DIM, head_depth=HEAD_DEPTH,
                                head_hidden=HEAD_HIDDEN, head_dropout=HEAD_DROPOUT,
                                use_ablation=use_ablation, svg_kwargs=svg_kwargs, tvg_kwargs=tvg_kwargs)
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=False))
    model.eval()
    stats = torch.load(stats_path, map_location="cpu", weights_only=False)
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    best_repeat = meta.get("repeat")
    return model, stats, best_repeat, tag


def pick_points_to_explain(tag, best_repeat, n_per_category=N_PER_CATEGORY, seed=EXPLAIN_SEED):
    """Reads the SPECIFIC repeat's own raw test-predictions file (the
    repeat whose weights are actually loaded, per best_model_meta.json)
    and samples up to n_per_category points per TP/TN/FP/FN category.
    Returns a list of {"point_id", "category"} dicts."""
    if best_repeat is None:
        print(f"  [{tag}] no best_model_meta.json repeat recorded -- skipping point selection.")
        return []
    pred_path = CHECKPOINT_DIR / f"{tag}_history" / f"repeat{best_repeat}_test_predictions.json"
    if not pred_path.exists():
        print(f"  [{tag}] {pred_path} not found -- skipping.")
        return []
    records = json.loads(pred_path.read_text())
    rng = random.Random(seed)
    by_category = {}
    for r in records:
        by_category.setdefault(r["category"], []).append(r)
    picked = []
    for cat, rows in sorted(by_category.items()):
        sample = rng.sample(rows, min(n_per_category, len(rows)))
        picked.extend({"point_id": r["point_id"], "category": cat} for r in sample)
    return picked

## Run both explanation techniques across scenarios A-F

In [ ]:
all_records = []

for scenario in ["A", "B", "C", "D", "E", "F"]:
    print(f"\n=== Scenario {scenario} ===")
    model, stats, best_repeat, tag = load_scenario_model(scenario)
    print(f"  loaded {tag} (best repeat={best_repeat})")

    points = pick_points_to_explain(tag, best_repeat)
    print(f"  explaining {len(points)} points: "
          f"{ {c: sum(1 for p in points if p['category']==c) for c in sorted(set(p['category'] for p in points))} }")

    for p in points:
        pid, cat = p["point_id"], p["category"]
        svg_raw = torch.load(SVG_DIR / f"{pid}.pt", weights_only=False)
        tvg_raw = torch.load(TVG_DIR / f"{pid}.pt", weights_only=False)
        svg_norm, tvg_norm = ds.apply_normalization(copy.deepcopy(svg_raw), copy.deepcopy(tvg_raw), stats)

        try:
            records = explain.explain_scenario_point(
                model, scenario, svg_norm, tvg_norm, point_id=pid, category=cat,
                gnnexplainer_epochs=GNNEXPLAINER_EPOCHS, gnnexplainer_lr=GNNEXPLAINER_LR)
            all_records.extend(records)
        except Exception as e:
            print(f"    !! failed to explain {pid} ({cat}): {type(e).__name__}: {e}")

print(f"\nTotal explained (point, branch) records: {len(all_records)}")

## Aggregate into one tidy CSV

In [ ]:
explain_df = explain.aggregate_explanations(all_records)
explain_df.to_csv(EXPLAIN_DIR / "explanations_capacity_revision.csv", index=False)
print(f"Saved {len(explain_df)} rows to {EXPLAIN_DIR / 'explanations_capacity_revision.csv'}")
display(explain_df.head(10))

## Sanity check: does GNNExplainer confirm the anchor node's "free pass"?

`pool_anchor` (this branch's readout) always concatenates the anchor
node's (`ego` for SVG branches, `incident` for TVG branches) own
embedding into the graph vector, by construction -- it can never be
"voted out" the way it could be in `08i`'s DGCNN readout. This cell
checks whether GNNExplainer's learned node-importance mask reflects
that: is the anchor type's mean importance high and consistent across
explained points, relative to other node types?

In [ ]:
anchor_types = {"A": "ego", "B": "incident", "C_svg": "ego", "C_tvg": "incident",
                 "D_svg": "ego", "D_tvg": "incident", "E_svg": "ego", "E_tvg": "incident",
                 "F": "incident"}  # F's anchor is "incident" per UnifiedEncoder's convention

gnne_node = explain_df[(explain_df["source"] == "gnnexplainer") & (explain_df["kind"] == "node")]
rows = []
for scen, anchor_nt in anchor_types.items():
    sub = gnne_node[gnne_node["scenario"] == scen]
    anchor_rows = sub[sub["type"] == anchor_nt]
    other_rows = sub[sub["type"] != anchor_nt]
    if len(anchor_rows) == 0:
        continue
    rows.append({
        "scenario": scen, "anchor_type": anchor_nt,
        "anchor_mean_importance": anchor_rows["mean_value"].mean(),
        "other_types_mean_importance": other_rows["mean_value"].mean() if len(other_rows) else float("nan"),
        "n_points": anchor_rows["point_id"].nunique(),
    })
anchor_summary_df = pd.DataFrame(rows)
anchor_summary_df.to_csv(EXPLAIN_DIR / "anchor_importance_summary.csv", index=False)
display(anchor_summary_df)

In [ ]:
print("08g explainability run complete.")
print(f"Explained {explain_df['point_id'].nunique()} unique points across "
      f"{explain_df['scenario'].nunique()} scenario/branch tags.")
print(f"See {EXPLAIN_DIR / 'explanations_capacity_revision.csv'} for the full per-point tidy table")
print("(GNNExplainer node/edge importance + raw GATv2 attention weights, side by side)")
print(f"{EXPLAIN_DIR / 'anchor_importance_summary.csv'} for the anchor-node sanity check,")
print(f"and {EXPLAIN_DIR / 'type_importance_summary.csv'} / 'type_importance_top5.csv' for the")
print("per-scheme (scenario) importance summary across ALL node/edge types.")

## Final report: per-scheme (scenario) importance summary

`explanations_capacity_revision.csv` reports importance one point at a
time -- useful for a case study, not for comparing schemes. This cell
folds it into two report tables that ARE directly comparable across
scenarios/branches:

- **`type_importance_summary.csv`** -- one row per `(scenario, kind,
  type)`, with `gnnexplainer` and `attention` as separate mean-importance
  columns (attention averaged across both `GATv2Conv` layers first).
  Answers "does `C_svg` lean on `signage` more than `A` does".
- **`type_importance_top5.csv`** -- for each `(scenario, source)`, the
  top 5 node/edge types by mean importance, ranked -- a skimmable
  summary of what each scheme actually attends to / needs.

In [ ]:
type_pivot_df, type_topn_df = explain.build_type_importance_report(explain_df, top_n=5)

type_pivot_df.to_csv(EXPLAIN_DIR / "type_importance_summary.csv", index=False)
type_topn_df.to_csv(EXPLAIN_DIR / "type_importance_top5.csv", index=False)

print(f"Saved {len(type_pivot_df)} rows to {EXPLAIN_DIR / 'type_importance_summary.csv'}")
print(f"Saved {len(type_topn_df)} rows to {EXPLAIN_DIR / 'type_importance_top5.csv'}")
display(type_pivot_df)
display(type_topn_df)